# Project 3 — Your First Backtest (and an honest reckoning)

This is the pivotal one. We take a classic, beloved beginner strategy — the
**moving-average crossover** — backtest it properly, and then put it head-to-head
against the laziest possible alternative: **buying and holding and doing nothing.**

The strategy: track a fast average (e.g. 50-day) and a slow average (200-day).
- When the fast average crosses **above** the slow one, the trend is up -> be **in** the stock.
- When it crosses **below**, the trend is down -> go to **cash** (sit out).

Sounds clever. Here's my honest prediction: after trading costs, it will most likely
**lose to doing nothing** — at least on return. Watching that happen to *your own code*
is the moment you stop trusting backtests blindly and start being a real quant.

> Run top to bottom with Shift+Enter. Kernel (top right) should say **quant**.

## Step 1 — Load data and pick a stock

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

prices = pd.read_csv("../data/prices.csv", index_col=0, parse_dates=True)

# ---- Change these and re-run to experiment ----
STOCK         = "WES.AX"   # try "FMG.AX" and "BHP.AX" too — the verdict changes!
SHORT_WINDOW  = 50         # fast moving average (days)
LONG_WINDOW   = 200        # slow moving average (days)
COST_PER_TRADE = 0.001     # 0.1% per trade (brokerage + slippage). Bump to 0.005 to see costs bite.

print(f"Backtesting a {SHORT_WINDOW}/{LONG_WINDOW} crossover on {STOCK}")
prices[STOCK].tail()

## Step 2 — Build the signal (and dodge the #1 beginner trap)

We compute the two moving averages, then a **signal**: 1 when the fast MA is above the
slow MA (be in the market), 0 when it's below (be in cash).

Then the single most important line in this whole notebook:

```python
data["position"] = data["signal"].shift(1)
```

**Why `.shift(1)`?** The signal on a given day is computed from that day's *closing* price.
You can't trade on a close you haven't seen yet — the earliest you can act is the *next* day.
Shifting the signal forward by one day enforces that. **Forget this line and your backtest
"sees the future"** (look-ahead bias) and will look fantastic for completely fake reasons.
This is exactly how people fool themselves into thinking they've struck gold.

In [ ]:
data = pd.DataFrame(index=prices.index)
data["price"] = prices[STOCK]
data["ret"]   = data["price"].pct_change()      # simple daily return

data["sma_short"] = data["price"].rolling(SHORT_WINDOW).mean()
data["sma_long"]  = data["price"].rolling(LONG_WINDOW).mean()

# signal: 1 = in the market, 0 = in cash
data["signal"] = (data["sma_short"] > data["sma_long"]).astype(int)

# act on the signal the NEXT day — this prevents look-ahead bias
data["position"] = data["signal"].shift(1)

data = data.dropna()   # drop the early rows where the 200-day MA doesn't exist yet
data[["price", "sma_short", "sma_long", "signal", "position"]].tail()

## Step 3 — Apply trading costs and build the two return streams

Every time the position flips (0->1 or 1->0) that's a trade, and trades cost money.
`position.diff().abs()` is 1 on days a trade happens, 0 otherwise. We subtract the cost
on those days. Then:

- **Strategy return** = (are we in the market?) x (the day's return) − (any trading cost)
- **Buy & Hold return** = just the day's return, every day, no trading.

In [ ]:
data["trade"] = data["position"].diff().abs().fillna(0)

# strategy: earn the return only when in-market, minus costs on trade days
data["strat_ret"] = data["position"] * data["ret"] - data["trade"] * COST_PER_TRADE
# buy & hold: always invested
data["bh_ret"] = data["ret"]

# turn daily returns into growth curves (compound them up)
data["strat_equity"] = (1 + data["strat_ret"]).cumprod()
data["bh_equity"]    = (1 + data["bh_ret"]).cumprod()

n_trades = int(data["trade"].sum())
print(f"The strategy made {n_trades} trades over the period.")

## Step 4 — The scorecard

In [ ]:
def metrics(ret):
    ret = ret.dropna()
    total  = (1 + ret).prod() - 1
    years  = len(ret) / 252
    cagr   = (1 + total) ** (1 / years) - 1
    vol    = ret.std() * np.sqrt(252)
    sharpe = (ret.mean() * 252) / vol if vol > 0 else np.nan
    eq     = (1 + ret).cumprod()
    maxdd  = (eq / eq.cummax() - 1).min()
    return pd.Series({"Total Return": total, "CAGR": cagr,
                      "Ann Vol": vol, "Sharpe": sharpe, "Max Drawdown": maxdd})

comp = pd.DataFrame({
    "Strategy (MA cross)": metrics(data["strat_ret"]),
    "Buy & Hold":          metrics(data["bh_ret"]),
})

# pretty-print: percentages where it makes sense, Sharpe as a plain number
show = comp.copy()
for row in ["Total Return", "CAGR", "Ann Vol", "Max Drawdown"]:
    show.loc[row] = (comp.loc[row] * 100).round(1).astype(str) + "%"
show.loc["Sharpe"] = comp.loc["Sharpe"].round(2)
show.loc["Num Trades"] = [n_trades, 0]
show

**Read this table honestly.** Three questions:
1. Did the strategy beat Buy & Hold on **Total Return**? (Often: no.)
2. Did it at least lower the **Max Drawdown** — i.e. give a smoother ride? (Sometimes: yes.)
3. Is its **Sharpe** higher? (That's the real "was it worth it?" test.)

A common, completely respectable outcome is: the strategy made *less money* but with a
*gentler ride*. Whether that trade is "better" depends on you — but notice it did **not**
hand you free money. The cleverness mostly bought you lower drawdown, not higher return.

## Step 5 — The money shot: strategy vs doing nothing

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data.index, data["bh_equity"],    label="Buy & Hold (do nothing)", lw=1.5)
plt.plot(data.index, data["strat_equity"], label="Strategy (MA crossover)", lw=1.5)
plt.title(f"{STOCK}: growth of $1 — strategy vs doing nothing")
plt.ylabel("Value ($, started at 1)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Step 6 — Where it traded (the MAs and the crossovers)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data.index, data["price"],     label="Price", lw=1, alpha=0.6)
plt.plot(data.index, data["sma_short"], label=f"{SHORT_WINDOW}-day MA", lw=1.2)
plt.plot(data.index, data["sma_long"],  label=f"{LONG_WINDOW}-day MA", lw=1.2)
# shade the periods when the strategy was IN the market
plt.fill_between(data.index, data["price"].min(), data["price"].max(),
                 where=data["position"] == 1, alpha=0.08, color="green",
                 label="In market")
plt.title(f"{STOCK}: price, moving averages, and when the strategy held")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
### The lesson

The crossover *feels* smart, and the chart in Step 6 makes it look reasonable — it gets
you out during big downtrends. But in a strongly rising stock it also yanks you out during
healthy dips and makes you **miss the rebound**, so you give back much of the upside. After
costs, "being clever" frequently loses to "doing nothing."

**This is the whole point of the project.** A backtest that beats buy-and-hold is easy to
*want*; a backtest you can *trust* is the hard part. The skill isn't finding a strategy that
wins — it's being rigorous enough to know, honestly, when yours doesn't.

### Now actually experiment (this is the real exercise)
- Set `STOCK = "FMG.AX"` and re-run. In that monster uptrend, watch Buy & Hold *demolish*
  the strategy — going to cash during FMG's dips was very expensive.
- Set `STOCK = "BHP.AX"` — a choppier ride. Does the strategy's lower drawdown start to look
  more worth it here?
- Bump `COST_PER_TRADE = 0.005` and re-run. Watch costs eat the strategy alive — and remember
  *real* retail costs on a small account are higher still.
- Try `SHORT_WINDOW = 20, LONG_WINDOW = 50` (faster, more trades). More trading = more costs.
  Does tinkering help, or just churn?

If you find a setting that *does* beat buy-and-hold — be suspicious, not excited. Ask: did I
just try 20 combinations and keep the one that happened to win? (That's overfitting, and it's
the subject of a later project.)